<a href="https://colab.research.google.com/github/Abduep53/Abduep53/blob/ai50%2Fprojects%2F2024%2Fx%2Fcrossword/Scalp_AI_FINAL_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Scalp AI — FINAL Colab Pipeline

This notebook is adapted to the exact Google Drive structure:

```text
AIHub_Scalp/
├── Training/
├── Validation/
└── Meta/
```

It recursively discovers ZIP files such as:

```text
[원천]피지과다_3.중증.zip
[원천]비듬_0.양호.zip
[원천]미세각질_2.중등도.zip
```

The notebook trains **three separate four-level classifiers**:

- `flaky` = 미세각질
- `oily` = 피지과다
- `dandruff` = 비듬

Each model predicts:

- `0` = healthy / none
- `1` = mild
- `2` = moderate
- `3` = severe

Three separate models are used because your downloaded source archives are already separated by condition and severity. The Raspberry Pi application runs all three models on the same microscope frame.

The notebook never extracts the full 50 GB dataset. It:

1. Lists ZIP contents.
2. Samples a balanced subset.
3. Copies one ZIP at a time to Colab local storage.
4. Extracts only selected images.
5. Resizes them immediately.
6. Deletes the temporary ZIP.
7. Trains and exports three quantized `.tflite` models.


In [1]:

# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os, shutil, subprocess, sys

MY_DRIVE = Path("/content/drive/MyDrive")

# Put the shortcut to AIHub_Scalp somewhere inside My Drive.
# The next cell finds it automatically.
print("Mounted:", MY_DRIVE)
print("Exists:", MY_DRIVE.exists())


Mounted at /content/drive
Mounted: /content/drive/MyDrive
Exists: True


In [2]:
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
# 2. Find the AIHub_Scalp shortcut automatically
# Recommended shortcut location:
# My Drive/MTSI_Scalp_Project/AIHub_Scalp

preferred_path = MY_DRIVE / "MTSI_Scalp_Project" / "AIHub_Scalp"
candidate_roots = []

if (
    preferred_path.is_dir()
    and (preferred_path / "Training").exists()
    and (preferred_path / "Validation").exists()
):
    candidate_roots.append(preferred_path)
else:
    # Fallback: search My Drive for another shortcut location.
    for path in MY_DRIVE.rglob("AIHub_Scalp"):
        if path.is_dir():
            has_training = (path / "Training").exists()
            has_validation = (path / "Validation").exists()
            if has_training and has_validation:
                candidate_roots.append(path)

print("Candidates found:")
for p in candidate_roots:
    print(" -", p)

if not candidate_roots:
    raise FileNotFoundError(
        "AIHub_Scalp was not found inside My Drive. "
        "In Google Drive: Shared with me -> right-click AIHub_Scalp -> "
        "Organize -> Add shortcut -> My Drive -> MTSI_Scalp_Project."
    )

DATA_ROOT = candidate_roots[0]
TRAIN_ZIP_ROOT = DATA_ROOT / "Training"
VALID_ZIP_ROOT = DATA_ROOT / "Validation"
META_ROOT = DATA_ROOT / "Meta"

# Save outputs in your own My Drive, not inside the shared dataset folder.
OUTPUT_ROOT = MY_DRIVE / "Scalp_AI_Output"
MODEL_ROOT = OUTPUT_ROOT / "models"
REPORT_ROOT = OUTPUT_ROOT / "reports"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

print("\nDATA_ROOT =", DATA_ROOT)
print("TRAIN_ZIP_ROOT =", TRAIN_ZIP_ROOT)
print("VALID_ZIP_ROOT =", VALID_ZIP_ROOT)
print("META_ROOT =", META_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)


Candidates found:
 - /content/drive/MyDrive/MTSI_Scalp_Project/AIHub_Scalp

DATA_ROOT = /content/drive/MyDrive/MTSI_Scalp_Project/AIHub_Scalp
TRAIN_ZIP_ROOT = /content/drive/MyDrive/MTSI_Scalp_Project/AIHub_Scalp/Training
VALID_ZIP_ROOT = /content/drive/MyDrive/MTSI_Scalp_Project/AIHub_Scalp/Validation
META_ROOT = /content/drive/MyDrive/MTSI_Scalp_Project/AIHub_Scalp/Meta
OUTPUT_ROOT = /content/drive/MyDrive/Scalp_AI_Output


In [6]:

# 3. Confirm GPU and local disk
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print()

subprocess.run(["nvidia-smi"], check=False)
print()
subprocess.run(["df", "-h", "/content", "/content/drive"], check=False)

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime -> Change runtime type -> GPU."
    )


TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]




In [7]:

# 4. Imports and configuration
from __future__ import annotations

import gc
import hashlib
import json
import math
import random
import re
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Deadline-friendly balanced subset.
# Start with these values. Increase only after the entire pipeline works.
TRAIN_PER_SEVERITY = 2500
TEST_PER_SEVERITY = 600

# Input size for MobileNetV2 on Raspberry Pi.
IMAGE_SIZE = 192
JPEG_QUALITY = 88
BATCH_SIZE = 32

CONDITION_PATTERNS = {
    "flaky": ["미세각질"],
    "oily": ["피지과다"],
    "dandruff": ["비듬"],
}

SEVERITY_WORDS = {
    0: "healthy",
    1: "mild",
    2: "moderate",
    3: "severe",
}

LOCAL_WORK = Path("/content/scalp_ai_work")
LOCAL_ZIP = LOCAL_WORK / "current.zip"
PREPARED_ROOT = LOCAL_WORK / "prepared"
LOCAL_WORK.mkdir(parents=True, exist_ok=True)

print("Configuration ready.")


Configuration ready.


In [8]:

# 5. Parse Korean ZIP filenames and recursively discover source archives

def parse_condition(zip_name: str):
    for condition, patterns in CONDITION_PATTERNS.items():
        if any(pattern in zip_name for pattern in patterns):
            return condition
    return None

def parse_severity(zip_name: str):
    # Examples:
    # [원천]피지과다_3.중증.zip
    # [원천]비듬_0.양호.zip
    match = re.search(r"[_-]([0-3])\s*\.", zip_name)
    if match:
        return int(match.group(1))

    # Fallback to Korean severity words.
    if "양호" in zip_name:
        return 0
    if "경증" in zip_name:
        return 1
    if "중등도" in zip_name:
        return 2
    if "중증" in zip_name:
        return 3
    return None

def is_source_zip(zip_name: str):
    # [원천] means source/original images.
    return "[원천]" in zip_name or "원천" in zip_name

zip_rows = []

for split_name, split_root in [
    ("training", TRAIN_ZIP_ROOT),
    ("validation", VALID_ZIP_ROOT),
]:
    all_zips = sorted(split_root.rglob("*.zip"))
    print(split_name, "ZIP count:", len(all_zips))

    for zip_path in all_zips:
        condition = parse_condition(zip_path.name)
        severity = parse_severity(zip_path.name)

        zip_rows.append({
            "split_source": split_name,
            "zip_path": str(zip_path),
            "zip_name": zip_path.name,
            "condition": condition,
            "severity": severity,
            "is_source": is_source_zip(zip_path.name),
            "compressed_gb": zip_path.stat().st_size / (1024 ** 3),
        })

zip_df = pd.DataFrame(zip_rows)

print("\nAll ZIP summary:")
display(zip_df.head(20))

relevant_zip_df = zip_df[
    zip_df["is_source"]
    & zip_df["condition"].notna()
    & zip_df["severity"].notna()
].copy()

print("\nRelevant source ZIPs:")
display(
    relevant_zip_df.groupby(
        ["split_source", "condition", "severity"]
    ).agg(
        zip_files=("zip_path", "count"),
        compressed_gb=("compressed_gb", "sum"),
    )
)

missing_buckets = []
for split_name in ["training", "validation"]:
    for condition in CONDITION_PATTERNS:
        for severity in range(4):
            exists = (
                (relevant_zip_df.split_source == split_name)
                & (relevant_zip_df.condition == condition)
                & (relevant_zip_df.severity == severity)
            ).any()
            if not exists:
                missing_buckets.append((split_name, condition, severity))

if missing_buckets:
    print("\nWARNING: missing condition/severity buckets:")
    for item in missing_buckets:
        print(item)
else:
    print("\nAll 24 expected buckets are present.")


training ZIP count: 24
validation ZIP count: 24

All ZIP summary:


,split_source,zip_path,zip_name,condition,severity,is_source,compressed_gb
0,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]미세각질_0.양호.zip,None,0,False,0.000151
1,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]미세각질_1.경증.zip,None,1,False,0.001270
2,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]미세각질_2.중등도.zip,None,2,False,0.001575
3,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]미세각질_3.중증.zip,None,3,False,0.000657
4,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]비듬_0.양호.zip,None,0,False,0.000151
5,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]비듬_1.경증.zip,None,1,False,0.004770
6,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]비듬_2.중등도.zip,None,2,False,0.002747
7,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]비듬_3.중증.zip,None,3,False,0.000652
8,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]피지과다_0.양호.zip,None,0,False,0.000151
9,training,/content/drive/MyDrive/MTSI_Scalp_Project/AIHu...,[라벨]피지과다_1.경증.zip,None,1,False,0.010427



Relevant source ZIPs:


,,,zip_files,compressed_gb
split_source,condition,severity,,



('training', 'flaky', 0)
('training', 'flaky', 1)
('training', 'flaky', 2)
('training', 'flaky', 3)
('training', 'oily', 0)
('training', 'oily', 1)
('training', 'oily', 2)
('training', 'oily', 3)
('training', 'dandruff', 0)
('training', 'dandruff', 1)
('training', 'dandruff', 2)
('training', 'dandruff', 3)
('validation', 'flaky', 0)
('validation', 'flaky', 1)
('validation', 'flaky', 2)
('validation', 'flaky', 3)
('validation', 'oily', 0)
('validation', 'oily', 1)
('validation', 'oily', 2)
('validation', 'oily', 3)
('validation', 'dandruff', 0)
('validation', 'dandruff', 1)
('validation', 'dandruff', 2)
('validation', 'dandruff', 3)


In [10]:
print("All ZIP rows:", len(zip_df))
print("Relevant source ZIP rows:", len(relevant_zip_df))

print("\nColumns:")
print(relevant_zip_df.columns.tolist())

print("\nFirst relevant ZIP files:")
display(
    relevant_zip_df[
        [
            "split_source",
            "condition",
            "severity",
            "zip_name",
            "zip_path",
            "compressed_gb",
        ]
    ].head(30)
)

All ZIP rows: 48
Relevant source ZIP rows: 0

Columns:
['split_source', 'zip_path', 'zip_name', 'condition', 'severity', 'is_source', 'compressed_gb']

First relevant ZIP files:


,split_source,condition,severity,zip_name,zip_path,compressed_gb


In [12]:
# Exact diagnosis of discovered ZIP archives

from pathlib import Path
from collections import Counter
import zipfile
import pandas as pd

print("=" * 90)
print("ZIP DIAGNOSTIC")
print("=" * 90)

print("All discovered ZIPs:", len(zip_df))
print("Relevant source ZIPs:", len(relevant_zip_df))

if relevant_zip_df.empty:
    print("\nERROR: relevant_zip_df is empty.")
    print("The filename parser did not recognize any source-image ZIP files.")

    print("\nFirst discovered ZIP names:")
    for name in zip_df["zip_name"].head(100):
        print(" -", repr(name))

    raise RuntimeError(
        "No relevant source ZIP files were recognized. "
        "Inspect the printed filenames and fix cell 5."
    )

diagnostic_rows = []

for idx, row in relevant_zip_df.reset_index(drop=True).iterrows():
    path = Path(row["zip_path"])

    result = {
        "index": idx + 1,
        "split": row["split_source"],
        "condition": row["condition"],
        "severity": row["severity"],
        "zip_name": path.name,
        "exists": path.exists(),
        "is_file": path.is_file(),
        "size_mb": (
            round(path.stat().st_size / (1024 ** 2), 2)
            if path.exists()
            else None
        ),
        "is_valid_zip": False,
        "members": None,
        "image_members": None,
        "nested_zip_members": None,
        "top_extensions": None,
        "sample_members": None,
        "error": None,
    }

    try:
        if not path.exists():
            raise FileNotFoundError(f"Path does not exist: {path}")

        result["is_valid_zip"] = zipfile.is_zipfile(path)

        if not result["is_valid_zip"]:
            raise zipfile.BadZipFile(
                "The file exists but is not recognized as a valid ZIP archive."
            )

        with zipfile.ZipFile(path, "r") as zf:
            infos = [info for info in zf.infolist() if not info.is_dir()]

            extension_counts = Counter(
                Path(info.filename).suffix.lower()
                for info in infos
            )

            image_infos = [
                info for info in infos
                if Path(info.filename).suffix.lower()
                in IMAGE_EXTENSIONS
            ]

            nested_zip_infos = [
                info for info in infos
                if Path(info.filename).suffix.lower() == ".zip"
            ]

            result["members"] = len(infos)
            result["image_members"] = len(image_infos)
            result["nested_zip_members"] = len(nested_zip_infos)
            result["top_extensions"] = dict(
                extension_counts.most_common(10)
            )
            result["sample_members"] = [
                info.filename for info in infos[:10]
            ]

    except Exception as exc:
        result["error"] = repr(exc)

    diagnostic_rows.append(result)

diagnostic_df = pd.DataFrame(diagnostic_rows)

display(
    diagnostic_df[
        [
            "index",
            "split",
            "condition",
            "severity",
            "zip_name",
            "exists",
            "size_mb",
            "is_valid_zip",
            "members",
            "image_members",
            "nested_zip_members",
            "top_extensions",
            "error",
        ]
    ]
)

print("\nDetailed samples from the first five archives:")

for row in diagnostic_rows[:5]:
    print("\n" + "-" * 90)
    print("ZIP:", row["zip_name"])
    print("Exists:", row["exists"])
    print("Valid ZIP:", row["is_valid_zip"])
    print("Members:", row["members"])
    print("Image members:", row["image_members"])
    print("Nested ZIP members:", row["nested_zip_members"])
    print("Extensions:", row["top_extensions"])
    print("Error:", row["error"])
    print("Sample members:")

    for member in row["sample_members"] or []:
        print("  ", repr(member))

diagnostic_df.to_csv(
    REPORT_ROOT / "zip_diagnostic.csv",
    index=False,
)

print("\nSaved diagnostic report:")
print(REPORT_ROOT / "zip_diagnostic.csv")

ZIP DIAGNOSTIC
All discovered ZIPs: 48
Relevant source ZIPs: 0

ERROR: relevant_zip_df is empty.
The filename parser did not recognize any source-image ZIP files.

First discovered ZIP names:
 - '[라벨]미세각질_0.양호.zip'
 - '[라벨]미세각질_1.경증.zip'
 - '[라벨]미세각질_2.중등도.zip'
 - '[라벨]미세각질_3.중증.zip'
 - '[라벨]비듬_0.양호.zip'
 - '[라벨]비듬_1.경증.zip'
 - '[라벨]비듬_2.중등도.zip'
 - '[라벨]비듬_3.중증.zip'
 - '[라벨]피지과다_0.양호.zip'
 - '[라벨]피지과다_1.경증.zip'
 - '[라벨]피지과다_2.중등도.zip'
 - '[라벨]피지과다_3.중증.zip'
 - '[원천]미세각질_0.양호.zip'
 - '[원천]미세각질_1.경증.zip'
 - '[원천]미세각질_2.중등도.zip'
 - '[원천]미세각질_3.중증.zip'
 - '[원천]비듬_0.양호.zip'
 - '[원천]비듬_1.zip'
 - '[원천]비듬_2.중등도.zip'
 - '[원천]비듬_3.중증.zip'
 - '[원천]피지과다_0.양호.zip'
 - '[원천]피지과다_1.경증.zip'
 - '[원천]피지과다_2.중등도.zip'
 - '[원천]피지과다_3.ᄌ

RuntimeError: No relevant source ZIP files were recognized. Inspect the printed filenames and fix cell 5.

In [13]:
# 5. Discover and classify ZIP archives robustly
# Handles Korean filenames stored in decomposed Unicode form.

from pathlib import Path
import pandas as pd
import unicodedata
import re

def normalize_unicode(text):
    """
    Convert decomposed Korean characters such as:
    원천
    into composed Korean:
    원천
    """
    return unicodedata.normalize("NFC", str(text))


def parse_zip_filename(zip_path):
    zip_path = Path(zip_path)

    original_name = zip_path.name
    normalized_name = normalize_unicode(original_name)
    normalized_lower = normalized_name.lower()

    # Determine whether this is source image data or label JSON data.
    is_source = (
        "[원천]" in normalized_name
        or "원천" in normalized_name
    )

    is_label = (
        "[라벨]" in normalized_name
        or "라벨" in normalized_name
    )

    # Map Korean dataset condition names to our English names.
    condition = None

    if "미세각질" in normalized_name:
        condition = "flaky"

    elif "비듬" in normalized_name:
        condition = "dandruff"

    elif "피지과다" in normalized_name:
        condition = "oily"

    # Extract severity from text after underscore.
    # Examples:
    # _0.양호.zip
    # _1.경증.zip
    # _2.중등도.zip
    # _3.중증.zip
    # _1.zip
    severity_match = re.search(
        r"_(\d)(?:\.|\.zip|$)",
        normalized_name,
        flags=re.IGNORECASE,
    )

    severity = None

    if severity_match:
        severity = int(severity_match.group(1))

    return {
        "zip_name": original_name,
        "normalized_zip_name": normalized_name,
        "condition": condition,
        "severity": severity,
        "is_source": is_source,
        "is_label": is_label,
    }


zip_rows = []

for split_source, zip_root in [
    ("training", TRAIN_ZIP_ROOT),
    ("validation", VALID_ZIP_ROOT),
]:
    zip_paths = sorted(zip_root.rglob("*.zip"))

    print(f"{split_source} ZIP count:", len(zip_paths))

    for zip_path in zip_paths:
        parsed = parse_zip_filename(zip_path)

        zip_rows.append({
            "split_source": split_source,
            "zip_path": str(zip_path),
            "zip_name": parsed["zip_name"],
            "normalized_zip_name": parsed["normalized_zip_name"],
            "condition": parsed["condition"],
            "severity": parsed["severity"],
            "is_source": parsed["is_source"],
            "is_label": parsed["is_label"],
            "compressed_gb": (
                zip_path.stat().st_size / (1024 ** 3)
            ),
        })


zip_df = pd.DataFrame(zip_rows)

if zip_df.empty:
    raise RuntimeError(
        "No ZIP files were found inside Training or Validation."
    )


print("\nAll ZIP summary:")

display(
    zip_df[
        [
            "split_source",
            "zip_name",
            "normalized_zip_name",
            "condition",
            "severity",
            "is_source",
            "is_label",
            "compressed_gb",
        ]
    ]
)


# Keep only source-image archives for selected conditions.
TARGET_CONDITIONS = {
    "flaky",
    "dandruff",
    "oily",
}

relevant_zip_df = zip_df[
    zip_df["is_source"]
    & zip_df["condition"].isin(TARGET_CONDITIONS)
    & zip_df["severity"].isin([0, 1, 2, 3])
].copy()

relevant_zip_df = relevant_zip_df.reset_index(drop=True)


print("\nRelevant source ZIPs:")

if relevant_zip_df.empty:
    print("NONE")
else:
    relevant_summary = (
        relevant_zip_df
        .groupby(
            [
                "split_source",
                "condition",
                "severity",
            ],
            dropna=False,
        )
        .agg(
            zip_files=("zip_path", "count"),
            compressed_gb=("compressed_gb", "sum"),
        )
        .reset_index()
    )

    display(relevant_summary)


# Verify all expected buckets.
expected_buckets = {
    (split_source, condition, severity)
    for split_source in ["training", "validation"]
    for condition in TARGET_CONDITIONS
    for severity in [0, 1, 2, 3]
}

found_buckets = set(
    relevant_zip_df[
        [
            "split_source",
            "condition",
            "severity",
        ]
    ].itertuples(index=False, name=None)
)

missing_buckets = sorted(
    expected_buckets - found_buckets
)

if missing_buckets:
    print("\nWARNING: missing condition/severity buckets:")

    for bucket in missing_buckets:
        print(bucket)
else:
    print("\nAll expected condition/severity buckets were found.")


print("\nFinal counts:")
print("All ZIP rows:", len(zip_df))
print("Relevant source ZIP rows:", len(relevant_zip_df))
print(
    "Relevant compressed size:",
    round(relevant_zip_df["compressed_gb"].sum(), 3),
    "GB",
)

training ZIP count: 24
validation ZIP count: 24

All ZIP summary:


,split_source,zip_name,normalized_zip_name,condition,severity,is_source,is_label,compressed_gb
0,training,[라벨]미세각질_0.양호.zip,[라벨]미세각질_0.양호.zip,flaky,0,False,True,0.000151
1,training,[라벨]미세각질_1.경증.zip,[라벨]미세각질_1.경증.zip,flaky,1,False,True,0.001270
2,training,[라벨]미세각질_2.중등도.zip,[라벨]미세각질_2.중등도.zip,flaky,2,False,True,0.001575
3,training,[라벨]미세각질_3.중증.zip,[라벨]미세각질_3.중증.zip,flaky,3,False,True,0.000657
4,training,[라벨]비듬_0.양호.zip,[라벨]비듬_0.양호.zip,dandruff,0,False,True,0.000151
5,training,[라벨]비듬_1.경증.zip,[라벨]비듬_1.경증.zip,dandruff,1,False,True,0.004770
6,training,[라벨]비듬_2.중등도.zip,[라벨]비듬_2.중등도.zip,dandruff,2,False,True,0.002747
7,training,[라벨]비듬_3.중증.zip,[라벨]비듬_3.중증.zip,dandruff,3,False,True,0.000652
8,training,[라벨]피지과다_0.양호.zip,[라벨]피지과다_0.양호.zip,oily,0,False,True,0.000151
9,training,[라벨]피지과다_1.경증.zip,[라벨]피지과다_1.경증.zip,oily,1,False,True,0.010427



Relevant source ZIPs:


,split_source,condition,severity,zip_files,compressed_gb
0,training,dandruff,0,1,0.119475
1,training,dandruff,1,1,3.837101
2,training,dandruff,2,1,2.241364
3,training,dandruff,3,1,0.542905
4,training,flaky,0,1,0.119475
5,training,flaky,1,1,0.993310
6,training,flaky,2,1,1.257034
7,training,flaky,3,1,0.540966
8,training,oily,0,1,0.119475
9,training,oily,1,1,6.379939



All expected condition/severity buckets were found.

Final counts:
All ZIP rows: 48
Relevant source ZIP rows: 24
Relevant compressed size: 29.194 GB


In [15]:

# 6. Read only ZIP central directories and build an image manifest
# This does not extract images.

manifest_rows = []
zip_errors = []

for idx, row in relevant_zip_df.reset_index(drop=True).iterrows():
    zip_path = Path(row["zip_path"])
    print(
        f"[{idx + 1}/{len(relevant_zip_df)}] Indexing:",
        row["split_source"], row["condition"], row["severity"], zip_path.name
    )

    try:
        with zipfile.ZipFile(zip_path) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                extension = Path(info.filename).suffix.lower()
                if extension not in IMAGE_EXTENSIONS:
                    continue

                image_stem = Path(info.filename).stem
                # Official naming starts with a de-identified subject ID.
                subject_id = image_stem.split("_")[0]

                manifest_rows.append({
                    "split_source": row["split_source"],
                    "condition": row["condition"],
                    "severity": int(row["severity"]),
                    "zip_path": str(zip_path),
                    "zip_name": zip_path.name,
                    "member": info.filename,
                    "image_stem": image_stem,
                    "subject_id": subject_id,
                    "uncompressed_bytes": info.file_size,
                })
    except Exception as exc:
        zip_errors.append({
            "zip_path": str(zip_path),
            "error": repr(exc),
        })

manifest = pd.DataFrame(manifest_rows)

if manifest.empty:
    raise RuntimeError(
        "No images were indexed. Confirm that the ZIPs contain JPG/PNG images."
    )

# Remove duplicate appearances of the same image within the same model/split.
before = len(manifest)
manifest = manifest.drop_duplicates(
    subset=["split_source", "condition", "image_stem"],
    keep="first",
).reset_index(drop=True)

print("\nIndexed rows before deduplication:", before)
print("Indexed unique rows:", len(manifest))
print("ZIP errors:", len(zip_errors))

if zip_errors:
    display(pd.DataFrame(zip_errors))

summary = manifest.groupby(
    ["split_source", "condition", "severity"]
).size().rename("images").reset_index()

display(summary)
summary.to_csv(REPORT_ROOT / "dataset_bucket_counts.csv", index=False)

# Detect a dangerous case: one image assigned to two severity levels
# within the same condition and source split.
conflicts = (
    pd.DataFrame(manifest_rows)
    .groupby(["split_source", "condition", "image_stem"])["severity"]
    .nunique()
)
conflicts = conflicts[conflicts > 1]

print("Severity conflicts:", len(conflicts))
if len(conflicts):
    display(conflicts.head(20))
    raise RuntimeError(
        "The same image appears under multiple severities for one condition. "
        "Stop and inspect the source ZIP structure."
    )


[1/24] Indexing: training flaky 0 [원천]미세각질_0.양호.zip
[2/24] Indexing: training flaky 1 [원천]미세각질_1.경증.zip
[3/24] Indexing: training flaky 2 [원천]미세각질_2.중등도.zip
[4/24] Indexing: training flaky 3 [원천]미세각질_3.중증.zip
[5/24] Indexing: training dandruff 0 [원천]비듬_0.양호.zip
[6/24] Indexing: training dandruff 1 [원천]비듬_1.zip
[7/24] Indexing: training dandruff 2 [원천]비듬_2.중등도.zip
[8/24] Indexing: training dandruff 3 [원천]비듬_3.중증.zip
[9/24] Indexing: training oily 0 [원천]피지과다_0.양호.zip
[10/24] Indexing: training oily 1 [원천]피지과다_1.경증.zip
[11/24] Indexing: training oily 2 [원천]피지과다_2.중등도.zip
[12/24] Indexing: training oily 3 [원천]피지과다_3.중증.zip
[13/24] Indexing: validation flaky 0 [원천]미세각질_0.양호.zip
[14/24] Indexing: validation flaky 1 [원천]미세각질_1.경증.zip
[15/24] Indexing: validation flaky 2 [원천]미세각질_2.중등도.zip
[16/24] Indexing: val

,split_source,condition,severity,images
0,training,dandruff,0,534
1,training,dandruff,1,16560
2,training,dandruff,2,9523
3,training,dandruff,3,2256
4,training,flaky,0,534
5,training,flaky,1,4435
6,training,flaky,2,5486
7,training,flaky,3,2284
8,training,oily,0,534
9,training,oily,1,28061


Severity conflicts: 0


In [16]:

# 7. Select a balanced subset from every condition/severity bucket

selected_parts = []

for split_name in ["training", "validation"]:
    cap = TRAIN_PER_SEVERITY if split_name == "training" else TEST_PER_SEVERITY

    for condition in CONDITION_PATTERNS:
        for severity in range(4):
            bucket = manifest[
                (manifest.split_source == split_name)
                & (manifest.condition == condition)
                & (manifest.severity == severity)
            ]

            if bucket.empty:
                print("EMPTY:", split_name, condition, severity)
                continue

            n = min(cap, len(bucket))
            sampled = bucket.sample(n=n, random_state=RANDOM_SEED + severity)
            selected_parts.append(sampled)

selected = pd.concat(selected_parts, ignore_index=True)

print("Selected total:", len(selected))
display(
    selected.groupby(
        ["split_source", "condition", "severity"]
    ).size().rename("selected").reset_index()
)

estimated_original_gb = selected.uncompressed_bytes.sum() / (1024 ** 3)
print("Selected original image bytes:", round(estimated_original_gb, 2), "GB")


Selected total: 29498


,split_source,condition,severity,selected
0,training,dandruff,0,534
1,training,dandruff,1,2500
2,training,dandruff,2,2500
3,training,dandruff,3,2256
4,training,flaky,0,534
5,training,flaky,1,2500
6,training,flaky,2,2500
7,training,flaky,3,2284
8,training,oily,0,534
9,training,oily,1,2500


Selected original image bytes: 6.85 GB


In [ ]:

# 8. Extract only selected images and resize immediately
# No full 50 GB extraction.

if PREPARED_ROOT.exists():
    shutil.rmtree(PREPARED_ROOT)
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)

prepared_rows = []
bad_rows = []

grouped = list(selected.groupby("zip_path"))
print("ZIP files needed for selected subset:", len(grouped))

for zip_index, (zip_path_string, group) in enumerate(grouped, 1):
    source_zip = Path(zip_path_string)

    if LOCAL_ZIP.exists():
        LOCAL_ZIP.unlink()

    print(f"\n[{zip_index}/{len(grouped)}] Copying:", source_zip.name)
    shutil.copy2(source_zip, LOCAL_ZIP)

    wanted = {row.member: row for row in group.itertuples(index=False)}

    try:
        with zipfile.ZipFile(LOCAL_ZIP) as zf:
            for member, row in wanted.items():
                try:
                    raw = zf.read(member)

                    with Image.open(BytesIO(raw)) as image:
                        image = ImageOps.exif_transpose(image).convert("RGB")

                        if image.width < 80 or image.height < 80:
                            raise ValueError(
                                f"Image too small: {image.width}x{image.height}"
                            )

                        # Preserve the central scalp field while producing a square tensor.
                        image = ImageOps.fit(
                            image,
                            (IMAGE_SIZE, IMAGE_SIZE),
                            method=Image.Resampling.LANCZOS,
                            centering=(0.5, 0.5),
                        )

                        unique_name = hashlib.sha1(
                            (
                                row.split_source
                                + "|"
                                + row.condition
                                + "|"
                                + str(row.severity)
                                + "|"
                                + row.image_stem
                            ).encode("utf-8")
                        ).hexdigest()[:20] + ".jpg"

                        output_dir = (
                            PREPARED_ROOT
                            / row.condition
                            / row.split_source
                            / str(row.severity)
                        )
                        output_dir.mkdir(parents=True, exist_ok=True)
                        output_path = output_dir / unique_name

                        image.save(
                            output_path,
                            format="JPEG",
                            quality=JPEG_QUALITY,
                            optimize=True,
                        )

                    prepared_rows.append({
                        "condition": row.condition,
                        "split_source": row.split_source,
                        "severity": int(row.severity),
                        "subject_id": str(row.subject_id),
                        "image_stem": row.image_stem,
                        "local_path": str(output_path),
                        "source_zip": str(source_zip),
                    })
                except Exception as exc:
                    bad_rows.append({
                        "zip": str(source_zip),
                        "member": member,
                        "error": repr(exc),
                    })
    finally:
        LOCAL_ZIP.unlink(missing_ok=True)
        gc.collect()

prepared = pd.DataFrame(prepared_rows)

print("\nPrepared valid images:", len(prepared))
print("Skipped bad images:", len(bad_rows))

if prepared.empty:
    raise RuntimeError("No images were prepared.")

display(
    prepared.groupby(
        ["condition", "split_source", "severity"]
    ).size().rename("prepared").reset_index()
)

prepared.to_csv(REPORT_ROOT / "prepared_manifest.csv", index=False)

if bad_rows:
    pd.DataFrame(bad_rows).to_csv(
        REPORT_ROOT / "bad_images.csv", index=False
    )

subprocess.run(["du", "-sh", str(PREPARED_ROOT)], check=False)
subprocess.run(["df", "-h", "/content"], check=False)


ZIP files needed for selected subset: 24

[1/24] Copying: [원천]미세각질_0.양호.zip

[2/24] Copying: [원천]미세각질_1.경증.zip

[3/24] Copying: [원천]미세각질_2.중등도.zip

[4/24] Copying: [원천]미세각질_3.중증.zip

[5/24] Copying: [원천]비듬_0.양호.zip

[6/24] Copying: [원천]비듬_1.zip

[7/24] Copying: [원천]비듬_2.중등도.zip

[8/24] Copying: [원천]비듬_3.중증.zip

[9/24] Copying: [원천]피지과다_0.양호.zip

[10/24] Copying: [원천]피지과다_1.경증.zip

[11/24] Copying: [원천]피지과다_2.중등도.zip

[12/24] Copying: [원천]피지과다_3.중증.zip

[13/24] Copying: [원천]미세각질_0.양호.zip

[14/24] Copying: [원천]미세각질_1.경증.zip


In [ ]:
8
# 9. Create train/validation/test tables
# - AI Hub Training folder -> internal train + internal validation
# - AI Hub Validation folder -> final test
# Internal split is grouped by subject ID.

from sklearn.model_selection import GroupShuffleSplit

condition_tables = {}

for condition in CONDITION_PATTERNS:
    condition_df = prepared[prepared.condition == condition].copy()

    source_train = condition_df[
        condition_df.split_source == "training"
    ].copy()

    source_test = condition_df[
        condition_df.split_source == "validation"
    ].copy()

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=RANDOM_SEED,
    )

    train_indices, val_indices = next(
        splitter.split(
            source_train,
            y=source_train.severity,
            groups=source_train.subject_id,
        )
    )

    train_df = source_train.iloc[train_indices].copy()
    val_df = source_train.iloc[val_indices].copy()
    test_df = source_test.copy()

    train_df["final_split"] = "train"
    val_df["final_split"] = "val"
    test_df["final_split"] = "test"

    final_df = pd.concat(
        [train_df, val_df, test_df],
        ignore_index=True,
    )

    # Exact-image overlap check between official folders.
    train_stems = set(
        final_df.loc[
            final_df.final_split.isin(["train", "val"]),
            "image_stem",
        ]
    )
    test_overlap = final_df[
        (final_df.final_split == "test")
        & (final_df.image_stem.isin(train_stems))
    ]

    if len(test_overlap):
        print(
            condition,
            "exact duplicate test images removed:",
            len(test_overlap),
        )
        final_df = final_df.drop(test_overlap.index).reset_index(drop=True)

    # Internal train/val subject leakage must be zero.
    train_subjects = set(
        final_df.loc[final_df.final_split == "train", "subject_id"]
    )
    val_subjects = set(
        final_df.loc[final_df.final_split == "val", "subject_id"]
    )
    assert train_subjects.isdisjoint(val_subjects)

    condition_tables[condition] = final_df

    print("\n", "=" * 72)
    print(condition.upper())
    display(
        final_df.groupby(
            ["final_split", "severity"]
        ).size().rename("images").reset_index()
    )
    print(
        "Train/val subject leakage:",
        len(train_subjects.intersection(val_subjects)),
    )


In [ ]:

# 10. TensorFlow input pipeline

from tensorflow import keras
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path, label):
    data = tf.io.read_file(path)
    image = tf.image.decode_jpeg(data, channels=3)
    image = tf.ensure_shape(image, [IMAGE_SIZE, IMAGE_SIZE, 3])
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.int32)

def make_dataset(df, training):
    paths = df.local_path.astype(str).to_numpy()
    labels = df.severity.astype("int32").to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=min(len(df), 10000),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

def class_weights_for(df):
    counts = (
        df.severity
        .value_counts()
        .reindex(range(4), fill_value=1)
        .astype(float)
    )
    weights = len(df) / (4.0 * counts)
    weights = weights.clip(upper=6.0)
    return {int(k): float(v) for k, v in weights.items()}


In [ ]:

# 11. Build a Pi-friendly MobileNetV2 classifier

def build_model():
    augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.06),
            layers.RandomZoom(0.08),
            layers.RandomContrast(0.12),
        ],
        name="augmentation",
    )

    inputs = keras.Input(
        shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        name="image",
    )

    x = augmentation(inputs)
    x = layers.Rescaling(
        1.0 / 127.5,
        offset=-1.0,
        name="mobilenet_preprocessing",
    )(x)

    backbone = keras.applications.MobileNetV2(
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        alpha=0.75,
        include_top=False,
        weights="imagenet",
    )
    backbone.trainable = False

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(4, name="severity_logits")(x)

    model = keras.Model(inputs, outputs)
    return model, backbone


In [ ]:

# 12. Train all three models
# Expected order: flaky -> oily -> dandruff

trained_models = {}
training_histories = {}

for condition in ["flaky", "oily", "dandruff"]:
    print("\n" + "#" * 80)
    print("TRAINING:", condition.upper())
    print("#" * 80)

    table = condition_tables[condition]
    train_df = table[table.final_split == "train"].copy()
    val_df = table[table.final_split == "val"].copy()

    train_ds = make_dataset(train_df, training=True)
    val_ds = make_dataset(val_df, training=False)

    model, backbone = build_model()

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
    )

    checkpoint = MODEL_ROOT / f"{condition}_best.keras"
    log_path = REPORT_ROOT / f"{condition}_training_log.csv"

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            checkpoint,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(log_path),
    ]

    class_weights = class_weights_for(train_df)
    print("Class weights:", class_weights)

    history_frozen = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        class_weight=class_weights,
        callbacks=callbacks,
    )

    # Fine-tune the last 25 backbone layers.
    backbone.trainable = True

    for layer in backbone.layers[:-25]:
        layer.trainable = False

    for layer in backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
    )

    start_epoch = len(history_frozen.epoch)

    history_finetune = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=start_epoch,
        epochs=start_epoch + 6,
        class_weight=class_weights,
        callbacks=callbacks,
    )

    best_model = keras.models.load_model(checkpoint)
    trained_models[condition] = best_model
    training_histories[condition] = {
        "frozen": history_frozen.history,
        "finetune": history_finetune.history,
    }

    del model, backbone, train_ds, val_ds
    gc.collect()
    tf.keras.backend.clear_session()

print("\nAll three models trained.")


In [ ]:

# 13. Evaluate on the untouched AI Hub Validation folder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

all_metrics = {}

for condition in ["flaky", "oily", "dandruff"]:
    print("\n" + "=" * 80)
    print("TEST:", condition.upper())
    print("=" * 80)

    model = trained_models[condition]
    table = condition_tables[condition]
    test_df = table[table.final_split == "test"].copy()
    test_ds = make_dataset(test_df, training=False)

    y_true = []
    logits_parts = []

    for batch_images, batch_labels in test_ds:
        logits = model.predict(batch_images, verbose=0)
        y_true.append(batch_labels.numpy())
        logits_parts.append(logits)

    y_true = np.concatenate(y_true)
    logits = np.concatenate(logits_parts)
    y_pred = logits.argmax(axis=1)

    accuracy = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(
        f1_score(y_true, y_pred, average="macro")
    )

    print("Accuracy:", accuracy)
    print("Macro F1:", macro_f1)
    print()
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            target_names=[
                "healthy",
                "mild",
                "moderate",
                "severe",
            ],
            digits=4,
            zero_division=0,
        )
    )
    print("Confusion matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
        )
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=[
            "healthy",
            "mild",
            "moderate",
            "severe",
        ],
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "classification_report": report,
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
        ).tolist(),
        "test_images": int(len(y_true)),
    }

    all_metrics[condition] = metrics

(REPORT_ROOT / "all_test_metrics.json").write_text(
    json.dumps(all_metrics, indent=2),
    encoding="utf-8",
)

print("\nSaved:", REPORT_ROOT / "all_test_metrics.json")


In [ ]:

# 14. Export dynamic and full-INT8 TFLite models

TFLITE_ROOT = MODEL_ROOT / "tflite"
TFLITE_ROOT.mkdir(parents=True, exist_ok=True)

export_summary = {}

for condition in ["flaky", "oily", "dandruff"]:
    print("\nExporting:", condition)
    model = trained_models[condition]

    table = condition_tables[condition]
    train_df = table[table.final_split == "train"].copy()
    representative_ds = make_dataset(
        train_df.sample(
            n=min(400, len(train_df)),
            random_state=RANDOM_SEED,
        ),
        training=False,
    )

    # Dynamic-range model: reliable fallback.
    dynamic_converter = tf.lite.TFLiteConverter.from_keras_model(
        model
    )
    dynamic_converter.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]
    dynamic_bytes = dynamic_converter.convert()

    dynamic_path = (
        TFLITE_ROOT / f"{condition}_dynamic.tflite"
    )
    dynamic_path.write_bytes(dynamic_bytes)

    def representative_dataset():
        count = 0
        for batch_images, _ in representative_ds:
            for image in batch_images:
                yield [
                    tf.expand_dims(
                        tf.cast(image, tf.float32),
                        axis=0,
                    )
                ]
                count += 1
                if count >= 300:
                    return

    int8_path = TFLITE_ROOT / f"{condition}_int8.tflite"
    int8_success = False
    int8_error = None

    try:
        int8_converter = (
            tf.lite.TFLiteConverter.from_keras_model(model)
        )
        int8_converter.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]
        int8_converter.representative_dataset = (
            representative_dataset
        )
        int8_converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]
        int8_converter.inference_input_type = tf.uint8
        int8_converter.inference_output_type = tf.uint8

        int8_bytes = int8_converter.convert()
        int8_path.write_bytes(int8_bytes)
        int8_success = True
    except Exception as exc:
        int8_error = repr(exc)
        print("INT8 conversion failed:", int8_error)

    export_summary[condition] = {
        "dynamic_model": dynamic_path.name,
        "dynamic_mb": dynamic_path.stat().st_size / (1024 ** 2),
        "int8_model": int8_path.name if int8_success else None,
        "int8_mb": (
            int8_path.stat().st_size / (1024 ** 2)
            if int8_success
            else None
        ),
        "int8_error": int8_error,
    }

print(json.dumps(export_summary, indent=2))


In [ ]:

# 15. Verify every exported TFLite model

def dequantize_output(array, detail):
    if detail["dtype"] in (np.uint8, np.int8):
        scale, zero_point = detail["quantization"]
        return (
            array.astype(np.float32) - zero_point
        ) * scale
    return array.astype(np.float32)

def run_tflite_one(model_path, image):
    interpreter = tf.lite.Interpreter(
        model_path=str(model_path),
        num_threads=4,
    )
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    batch = np.expand_dims(
        image.astype(np.float32),
        axis=0,
    )

    if input_detail["dtype"] in (np.uint8, np.int8):
        scale, zero_point = input_detail["quantization"]
        batch = np.round(
            batch / scale + zero_point
        ).astype(input_detail["dtype"])
    else:
        batch = batch.astype(input_detail["dtype"])

    interpreter.set_tensor(
        input_detail["index"],
        batch,
    )
    interpreter.invoke()

    output = interpreter.get_tensor(
        output_detail["index"]
    )
    output = dequantize_output(
        output,
        output_detail,
    )
    return output[0]

verification = {}

for condition in ["flaky", "oily", "dandruff"]:
    table = condition_tables[condition]
    sample_row = table[
        table.final_split == "test"
    ].iloc[0]

    with Image.open(sample_row.local_path) as image:
        image_array = np.asarray(
            image.convert("RGB"),
            dtype=np.float32,
        )

    condition_result = {}

    for suffix in ["int8", "dynamic"]:
        model_path = (
            TFLITE_ROOT
            / f"{condition}_{suffix}.tflite"
        )
        if not model_path.exists():
            continue

        logits = run_tflite_one(
            model_path,
            image_array,
        )
        prediction = int(np.argmax(logits))

        condition_result[suffix] = {
            "true": int(sample_row.severity),
            "predicted": prediction,
            "logits": logits.tolist(),
        }

        print(
            condition,
            suffix,
            "true=",
            int(sample_row.severity),
            "predicted=",
            prediction,
        )

    verification[condition] = condition_result

(REPORT_ROOT / "tflite_verification.json").write_text(
    json.dumps(verification, indent=2),
    encoding="utf-8",
)


In [ ]:

# 16. Build the final Raspberry Pi model package

PACKAGE_DIR = Path("/content/scalp_pi_models")
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True)

chosen_models = {}

for condition in ["flaky", "oily", "dandruff"]:
    int8_path = TFLITE_ROOT / f"{condition}_int8.tflite"
    dynamic_path = (
        TFLITE_ROOT / f"{condition}_dynamic.tflite"
    )

    chosen = int8_path if int8_path.exists() else dynamic_path
    destination = PACKAGE_DIR / f"{condition}.tflite"
    shutil.copy2(chosen, destination)
    chosen_models[condition] = chosen.name

metadata = {
    "project": "UPenn M&TSI Scalp AI",
    "input_size": [IMAGE_SIZE, IMAGE_SIZE],
    "conditions": [
        "flaky",
        "oily",
        "dandruff",
    ],
    "severity_levels": [
        "healthy",
        "mild",
        "moderate",
        "severe",
    ],
    "condition_mapping": {
        "flaky": "미세각질",
        "oily": "피지과다",
        "dandruff": "비듬",
    },
    "confidence_threshold": 0.55,
    "chosen_source_models": chosen_models,
    "medical_disclaimer": (
        "Educational wellness prototype only. "
        "It does not diagnose disease and does not "
        "replace a qualified healthcare professional."
    ),
}

(PACKAGE_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

shutil.copy2(
    REPORT_ROOT / "all_test_metrics.json",
    PACKAGE_DIR / "all_test_metrics.json",
)

package_base = OUTPUT_ROOT / "scalp_pi_models"
archive_path = shutil.make_archive(
    str(package_base),
    "zip",
    root_dir=PACKAGE_DIR,
)

print("Final package:", archive_path)
print("Files:")
for path in sorted(PACKAGE_DIR.iterdir()):
    print(
        " -",
        path.name,
        round(path.stat().st_size / (1024 ** 2), 2),
        "MB",
    )


In [ ]:

# 17. Download the model package to your Mac
from google.colab import files

files.download(
    str(OUTPUT_ROOT / "scalp_pi_models.zip")
)
